# Unigram Language Model Evaluation

This notebook evaluates the Unigram Language Model (Dirichlet smoothing) across different values of `mu`.

It uses the same 25 queries and evaluation metrics as the BM25 evaluation for fair comparison.

In [1]:
import sys
sys.path.append("../")

import math
import importlib
import numpy as np
import pandas as pd
from itertools import product
import os

import src.query_data
importlib.reload(src.query_data)

from src.query_data import corpus_df, corpus, processed_corpus, query, query_prepro
from src.models import UnigramLM

Columns: ['category', 'filename', 'title', 'content']


[nltk_data] Downloading package stopwords to C:\Users\nm-
[nltk_data]     ca\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\nm-
[nltk_data]     ca\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Columns: ['category', 'filename', 'title', 'content']


[nltk_data] Downloading package stopwords to C:\Users\nm-
[nltk_data]     ca\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\nm-
[nltk_data]     ca\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# Cache _collection_prob to avoid recomputing the same term's probability thousands of times
# This does NOT change results — same formula, just stores the answer after first call per term
_original_init = UnigramLM.__init__

def _patched_init(self, corpus, mu):
    _original_init(self, corpus, mu)
    self._coll_prob_cache = {}

def _cached_collection_prob(self, q_term):
    if q_term in self._coll_prob_cache:
        return self._coll_prob_cache[q_term]
    freq = 0
    for doc in self.corpus:
        freq += doc.count(q_term)
    prob = freq / self.collection_len
    self._coll_prob_cache[q_term] = prob
    return prob

UnigramLM.__init__ = _patched_init
UnigramLM._collection_prob = _cached_collection_prob

print("UnigramLM patched with collection probability caching")

UnigramLM patched with collection probability caching


In [3]:
assert len(corpus) == len(processed_corpus), "Corpus and processed_corpus lengths do not match"
assert len(query) == len(query_prepro), "Query and query_prepro lengths do not match"
assert "category" in corpus_df.columns, "corpus_df must contain a category column"
assert "category" in query.columns, "query must contain a category column"

print("Documents:", len(corpus))
print("Queries:", len(query))

Documents: 2225
Queries: 25


In [4]:
def average_precision(relevance):
    """
    relevance: list of 0/1 values in ranked order
    """
    num_rel = sum(relevance)
    if num_rel == 0:
        return 0.0

    precisions = []
    rel_found = 0

    for i, rel in enumerate(relevance, start=1):
        if rel == 1:
            rel_found += 1
            precisions.append(rel_found / i)

    return sum(precisions) / num_rel


def precision_at_k(relevance, k=10):
    if len(relevance) == 0:
        return 0.0
    return sum(relevance[:k]) / k


def dcg_at_k(relevance, k=10):
    relevance = relevance[:k]
    return sum(rel / math.log2(i + 2) for i, rel in enumerate(relevance))


def ndcg_at_k(relevance, k=10):
    dcg = dcg_at_k(relevance, k)
    ideal = sorted(relevance, reverse=True)
    idcg = dcg_at_k(ideal, k)
    return dcg / idcg if idcg > 0 else 0.0

In [5]:
def evaluate_unigram_model(
    model,
    query_df,
    query_prepro_list,
    ndcg_k=10,
    max_queries=None
):
    ap_scores = []
    p10_scores = []
    ndcg_scores = []
    per_query_results = []

    n_queries = len(query_df) if max_queries is None else min(max_queries, len(query_df))

    for i in range(n_queries):
        row = query_df.iloc[i]
        raw_query = row["query"]
        relevant_category = row["category"]

        ranked_results = model.rank(query_prepro_list[i])

        relevance = ranked_results["category"].eq(relevant_category).astype(int).tolist()

        ap = average_precision(relevance)
        p10 = precision_at_k(relevance, k=10)
        ndcg = ndcg_at_k(relevance, k=ndcg_k)

        ap_scores.append(ap)
        p10_scores.append(p10)
        ndcg_scores.append(ndcg)

        per_query_results.append({
            "query": raw_query,
            "category": relevant_category,
            "AP": ap,
            "Precision@10": p10,
            f"nDCG@{ndcg_k}": ndcg
        })

    summary = {
        "MAP": round(float(np.mean(ap_scores)), 4),
        "Precision@10": round(float(np.mean(p10_scores)), 4),
        f"nDCG@{ndcg_k}": round(float(np.mean(ndcg_scores)), 4)
    }

    per_query_df = pd.DataFrame(per_query_results)
    return summary, per_query_df

## Mu Grid Search

Evaluation is done in batches as it takes long. Run the cells below sequentially — results are appended to the same CSV.

In [6]:
mu_values = [100, 500, 1000, 1500, 2000, 2500, 3000, 5000]

all_results = []

for mu in mu_values:
    print(f"Evaluating mu={mu}...")
    model = UnigramLM(corpus=processed_corpus, mu=mu)

    summary, per_query = evaluate_unigram_model(
        model=model,
        query_df=query,
        query_prepro_list=query_prepro,
        ndcg_k=10,
        max_queries=None
    )

    print(f"  mu={mu} => MAP={summary['MAP']}, P@10={summary['Precision@10']}, nDCG@10={summary['nDCG@10']}")

    queries_ranked = per_query.sort_values("AP", ascending=False).reset_index(drop=True)
    top_5_queries = queries_ranked.head(5).copy()

    for _, row in top_5_queries.iterrows():
        all_results.append({
            "mu": mu,
            "MAP": summary["MAP"],
            "Precision@10_overall": summary["Precision@10"],
            "nDCG@10_overall": summary["nDCG@10"],
            "query": row["query"],
            "AP": row["AP"],
            "Precision@10_query": row["Precision@10"],
            "nDCG@10_query": row["nDCG@10"]
        })

results_df = pd.DataFrame(all_results)
results_df.drop_duplicates(inplace=True)

file_path = "../data/archive-5/unigram_evaluation_results_with_top5_queries.csv"

if os.path.exists(file_path):
    existing_df = pd.read_csv(file_path)
    combined_df = pd.concat([existing_df, results_df], ignore_index=True)
    combined_df.drop_duplicates(subset=["mu", "query"], inplace=True)
    combined_df.to_csv(file_path, index=False)
else:
    results_df.to_csv(file_path, index=False)

print("\nSaved: unigram_evaluation_results_with_top5_queries.csv")

Evaluating mu=100...
  mu=100 => MAP=0.5758, P@10=0.932, nDCG@10=0.9459
Evaluating mu=500...
  mu=500 => MAP=0.5785, P@10=0.92, nDCG@10=0.9349
Evaluating mu=1000...
  mu=1000 => MAP=0.5778, P@10=0.912, nDCG@10=0.9171
Evaluating mu=1500...
  mu=1500 => MAP=0.5767, P@10=0.912, nDCG@10=0.9173
Evaluating mu=2000...
  mu=2000 => MAP=0.5758, P@10=0.916, nDCG@10=0.9197
Evaluating mu=2500...
  mu=2500 => MAP=0.575, P@10=0.92, nDCG@10=0.9191
Evaluating mu=3000...
  mu=3000 => MAP=0.5741, P@10=0.92, nDCG@10=0.9177
Evaluating mu=5000...
  mu=5000 => MAP=0.572, P@10=0.928, nDCG@10=0.9192

Saved: unigram_evaluation_results_with_top5_queries.csv


In [7]:
# Quick preview of results
preview = pd.read_csv("../data/archive-5/unigram_evaluation_results_with_top5_queries.csv")
summary_preview = (
    preview.groupby("mu", as_index=False)
    .agg({"MAP": "first", "Precision@10_overall": "first", "nDCG@10_overall": "first"})
    .sort_values("MAP", ascending=False)
)
print("Summary by mu:")
display(summary_preview)

Summary by mu:


,mu,MAP,Precision@10_overall,nDCG@10_overall
1,500,0.5785,0.920,0.9349
2,1000,0.5778,0.912,0.9171
3,1500,0.5767,0.912,0.9173
0,100,0.5758,0.932,0.9459
4,2000,0.5758,0.916,0.9197
5,2500,0.5750,0.920,0.9191
6,3000,0.5741,0.920,0.9177
7,5000,0.5720,0.928,0.9192
